#  EV Scenario Generation

This notebook builds synthetic Keele EV charging scenarios.

## Purpose

Use cleaned ACN sessions to create synthetic EV charging sessions for Keele surplus days.

ACN provides typical workplace charging patterns. Keele provides the surplus dates and campus limits.

## Keele EV assumptions

These are synthetic Keele EV scenarios, not real Keele charger records.

- ACN supplies arrival times, departure times, dwell time, energy and charger power.
- Keele supplies the surplus-rich days and campus objective.
- Fleet size, participation, efficiency and site power limits are assumptions.
- Energy is clipped only when needed to keep a session feasible.

In [1]:
from pathlib import Path
import json
import math
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either the root or Notebooks directory."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing Data/ and Notebooks/.")


ROOT = find_project_root()
DATA_DIR = ROOT / "Data"
RESULTS_DIR = DATA_DIR / "Results"
PROCESSED_DIR = DATA_DIR / "Processed"
FIGURES_DIR = ROOT / "Figures"
MODELS_DIR = ROOT / "Models" / "Forecasting"

for directory in [RESULTS_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INTERVAL_MINUTES = 5
INTERVAL_HOURS = INTERVAL_MINUTES / 60
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print(f"Project root: {ROOT}")

Project root: /Users/mac/Send-Ev-Project/SEND-EV-Project


In [2]:
def save_table(df: pd.DataFrame, path: Path) -> Path:
    """Save Parquet where available, otherwise save CSV with the same stem."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(path, index=False)
            print(f"Saved: {path.relative_to(ROOT)}")
            return path
        except (ImportError, ModuleNotFoundError) as exc:
            csv_path = path.with_suffix(".csv")
            df.to_csv(csv_path, index=False)
            print(f"Parquet engine unavailable ({exc.__class__.__name__}); saved CSV: {csv_path.relative_to(ROOT)}")
            return csv_path
    df.to_csv(path, index=False)
    print(f"Saved: {path.relative_to(ROOT)}")
    return path


def read_table(path: Path, parse_dates: list[str] | None = None) -> pd.DataFrame:
    """Read a Parquet/CSV output regardless of which format was written."""
    candidates = [path]
    if path.suffix.lower() == ".parquet":
        candidates.append(path.with_suffix(".csv"))
    elif path.suffix.lower() == ".csv":
        candidates.append(path.with_suffix(".parquet"))

    for candidate in candidates:
        if candidate.exists():
            if candidate.suffix.lower() == ".parquet":
                return pd.read_parquet(candidate)
            return pd.read_csv(candidate, parse_dates=parse_dates)
    raise FileNotFoundError(f"None of these files exists: {candidates}")

In [3]:
ACN_PROCESSED_DIR = DATA_DIR / "ACN" / "processed"
acn = read_table(ACN_PROCESSED_DIR / "acn_sessions_cleaned.parquet")
send = read_table(PROCESSED_DIR / "send_solcast_surplus_5min.parquet", parse_dates=["DateTime"])

for column in ["arrival_utc", "departure_utc", "arrival_local", "departure_local"]:
    if column in acn.columns:
        acn[column] = pd.to_datetime(acn[column], errors="coerce", utc=True)
send["DateTime"] = pd.to_datetime(send["DateTime"], errors="coerce")

if "overnight" in acn.columns and acn["overnight"].dtype == object:
    acn["overnight"] = acn["overnight"].astype(str).str.lower().map({"true": True, "false": False}).fillna(False)

print(f"ACN sessions: {len(acn):,}")
print(f"SEND records: {len(send):,}")

ACN sessions: 28,924
SEND records: 193,247


## 1. Scenario configuration

In [4]:
# Add more values later for the full sensitivity test.
FLEET_SIZES = [10, 25, 50, 100]
PARTICIPATION_RATES = [0.25, 0.50, 0.75, 1.00]
SITE_SCENARIOS = ["pooled", "caltech", "jpl", "office001"]
RANDOM_SEEDS = [42, 43]
N_KEELE_DAYS = 10
CHARGING_EFFICIENCY = 0.90

# Assumed Keele site power limits.
SITE_LIMIT_BY_FLEET = {
    10: 50.0,
    25: 100.0,
    50: 200.0,
    100: 350.0,
}

# Pick Keele days with the most surplus.
send["date"] = send["DateTime"].dt.normalize()
daily = (
    send.groupby("date", as_index=False)
    .agg(
        surplus_energy_kwh=("surplus_kwh", "sum"),
        peak_surplus_kw=("surplus_kw", "max"),
        valid_intervals=("surplus_kw", "count"),
        total_intervals=("DateTime", "size"),
    )
)
daily["data_completeness"] = daily["valid_intervals"] / daily["total_intervals"]
daily = daily[daily["data_completeness"] >= 0.95].sort_values("surplus_energy_kwh", ascending=False)
selected_days = daily.head(N_KEELE_DAYS)["date"].tolist()
display(daily.head(N_KEELE_DAYS))

,date,surplus_energy_kwh,peak_surplus_kw,valid_intervals,total_intervals,data_completeness
493,2023-07-07,12713.815756,3015.486381,288,288,1.0
495,2023-07-09,10552.383127,3500.058734,288,288,1.0
117,2022-06-26,9023.666000,1376.591000,288,288,1.0
488,2023-07-02,8810.713357,2373.734168,288,288,1.0
74,2022-05-14,8555.557000,1523.840000,288,288,1.0
402,2023-04-07,8399.886250,1488.962000,288,288,1.0
417,2023-04-22,8398.182139,2291.222320,288,288,1.0
551,2023-09-03,8332.003167,1568.537000,288,288,1.0
470,2023-06-14,7775.247750,1544.163000,288,288,1.0
137,2022-07-16,7744.069750,1516.704000,288,288,1.0


## 2. Build scenario sessions

In [5]:
def minute_of_day(value) -> int:
    if pd.isna(value):
        return 0
    return int(value)


def build_scenario(
    pool: pd.DataFrame,
    keele_day: pd.Timestamp,
    fleet_size: int,
    participation_rate: float,
    source_site: str,
    seed: int,
) -> tuple[pd.DataFrame, dict]:
    day_type = "weekend" if keele_day.dayofweek >= 5 else "weekday"
    candidates = pool[pool["day_type"].eq(day_type)].copy()
    if candidates.empty:
        candidates = pool.copy()

    participant_count = max(1, int(round(fleet_size * participation_rate)))
    sampled = candidates.sample(
        n=participant_count,
        replace=participant_count > len(candidates),
        random_state=seed,
    ).reset_index(drop=True)

    rows = []
    clipped_energy_count = 0
    for vehicle_number, row in sampled.iterrows():
        arrival_minute = minute_of_day(row["arrival_minute"])
        departure_minute = minute_of_day(row["departure_minute"])

        arrival = keele_day.normalize() + pd.Timedelta(minutes=arrival_minute)
        departure = keele_day.normalize() + pd.Timedelta(minutes=departure_minute)
        if bool(row.get("overnight", False)) or departure <= arrival:
            departure += pd.Timedelta(days=1)

        # Cap unusual sessions so tests stay practical.
        departure = min(departure, arrival + pd.Timedelta(hours=48))
        dwell_hours = (departure - arrival).total_seconds() / 3600
        max_power_kw = float(row["max_power_kw"])
        original_energy = float(row["energy_required_kwh"])
        maximum_deliverable = max_power_kw * CHARGING_EFFICIENCY * dwell_hours
        energy_required = min(original_energy, maximum_deliverable)
        energy_was_clipped = energy_required < original_energy - 1e-6
        clipped_energy_count += int(energy_was_clipped)

        rows.append({
            "vehicle_id": f"EV{vehicle_number + 1:03d}",
            "source_session_id": row["session_id"],
            "source_site": row["site"],
            "scenario_site": source_site,
            "arrival_time": arrival,
            "departure_time": departure,
            "energy_required_kwh": energy_required,
            "original_energy_required_kwh": original_energy,
            "energy_clipped_for_feasibility": energy_was_clipped,
            "max_power_kw": max_power_kw,
            "efficiency": CHARGING_EFFICIENCY,
            "fleet_size": fleet_size,
            "participation_rate": participation_rate,
            "participant_count": participant_count,
            "site_limit_kw": SITE_LIMIT_BY_FLEET[fleet_size],
            "keele_date": keele_day.normalize(),
            "day_type": day_type,
            "random_seed": seed,
        })

    scenario_id = (
        f"{keele_day:%Y%m%d}_{source_site}_F{fleet_size}_P{int(participation_rate*100):03d}_S{seed}"
    )
    scenario = pd.DataFrame(rows)
    scenario.insert(0, "scenario_id", scenario_id)

    summary = {
        "scenario_id": scenario_id,
        "keele_date": keele_day.normalize(),
        "scenario_site": source_site,
        "fleet_size": fleet_size,
        "participation_rate": participation_rate,
        "participant_count": participant_count,
        "random_seed": seed,
        "site_limit_kw": SITE_LIMIT_BY_FLEET[fleet_size],
        "total_energy_required_kwh": scenario["energy_required_kwh"].sum(),
        "median_dwell_hours": ((scenario["departure_time"] - scenario["arrival_time"]).dt.total_seconds() / 3600).median(),
        "energy_clipped_sessions": clipped_energy_count,
    }
    return scenario, summary


scenario_frames = []
summary_rows = []
for day in selected_days:
    day = pd.Timestamp(day)
    for site in SITE_SCENARIOS:
        site_pool = acn if site == "pooled" else acn[acn["site"].eq(site)]
        if site_pool.empty:
            print(f"Skipping empty site pool: {site}")
            continue
        for fleet_size in FLEET_SIZES:
            for participation_rate in PARTICIPATION_RATES:
                for seed in RANDOM_SEEDS:
                    scenario, summary = build_scenario(
                        site_pool, day, fleet_size, participation_rate, site, seed
                    )
                    scenario_frames.append(scenario)
                    summary_rows.append(summary)

scenarios = pd.concat(scenario_frames, ignore_index=True)
scenario_summary = pd.DataFrame(summary_rows)
print(f"Generated {scenario_summary['scenario_id'].nunique():,} scenarios and {len(scenarios):,} participating vehicle records")
display(scenario_summary.head())

Generated 1,280 scenarios and 36,960 participating vehicle records


,scenario_id,keele_date,scenario_site,fleet_size,participation_rate,participant_count,random_seed,site_limit_kw,total_energy_required_kwh,median_dwell_hours,energy_clipped_sessions
0,20230707_pooled_F10_P025_S42,2023-07-07,pooled,10,0.25,2,42,50.0,14.725,2.966667,0
1,20230707_pooled_F10_P025_S43,2023-07-07,pooled,10,0.25,2,43,50.0,16.763,7.741667,0
2,20230707_pooled_F10_P050_S42,2023-07-07,pooled,10,0.50,5,42,50.0,64.482,3.633333,2
3,20230707_pooled_F10_P050_S43,2023-07-07,pooled,10,0.50,5,43,50.0,39.417,7.533333,2
4,20230707_pooled_F10_P075_S42,2023-07-07,pooled,10,0.75,8,42,50.0,96.707,6.016667,2


## 3. Validate scenario feasibility

In [6]:
scenarios["dwell_hours"] = (
    scenarios["departure_time"] - scenarios["arrival_time"]
).dt.total_seconds() / 3600
scenarios["maximum_deliverable_kwh"] = (
    scenarios["max_power_kw"] * scenarios["efficiency"] * scenarios["dwell_hours"]
)
scenarios["individually_feasible"] = (
    scenarios["energy_required_kwh"] <= scenarios["maximum_deliverable_kwh"] + 1e-6
)

assert scenarios["departure_time"].gt(scenarios["arrival_time"]).all()
assert scenarios["energy_required_kwh"].gt(0).all()
assert scenarios["individually_feasible"].all()

validation = (
    scenarios.groupby("scenario_id", as_index=False)
    .agg(
        vehicles=("vehicle_id", "size"),
        feasible_share=("individually_feasible", "mean"),
        total_energy_kwh=("energy_required_kwh", "sum"),
        earliest_arrival=("arrival_time", "min"),
        latest_departure=("departure_time", "max"),
    )
)
display(validation.head())

,scenario_id,vehicles,feasible_share,total_energy_kwh,earliest_arrival,latest_departure
0,20220514_caltech_F100_P025_S42,25,1.0,226.104557,2022-05-14 07:48:00,2022-05-15 10:06:00
1,20220514_caltech_F100_P025_S43,25,1.0,275.408000,2022-05-14 08:33:00,2022-05-15 18:33:00
2,20220514_caltech_F100_P050_S42,50,1.0,479.710557,2022-05-14 02:41:00,2022-05-15 16:11:00
3,20220514_caltech_F100_P050_S43,50,1.0,606.443000,2022-05-14 08:13:00,2022-05-15 18:33:00
4,20220514_caltech_F100_P075_S42,75,1.0,745.480961,2022-05-14 02:41:00,2022-05-15 16:11:00


## 4. Save outputs

In [7]:
save_table(scenarios, DATA_DIR / "ACN" / "processed" / "keele_ev_scenarios.parquet")
save_table(scenario_summary, RESULTS_DIR / "scenario_summary.csv")

Parquet engine unavailable (ImportError); saved CSV: Data/ACN/processed/keele_ev_scenarios.csv
Saved: Data/Results/scenario_summary.csv


PosixPath('/Users/mac/Send-Ev-Project/SEND-EV-Project/Data/Results/scenario_summary.csv')

## Interpretation

These are synthetic Keele EV charging profiles. Report the fleet size, participation, efficiency, site-capacity and sampling assumptions with the results.